# 3단계: 모델별 하이퍼파라미터 정밀 튜닝

2단계에서 비교한 다섯 실제 모델을 더 넓고 촘촘한 범위에서 튜닝한다. Dummy는 조정할 값이 없어 제외한다.

- 공통 입력: `Unknown`을 포함한 원본 범주형 13개
- LR·NB·ExtraTrees: 모델 파이프라인 내부 원핫 인코딩
- CatBoost·TabICL: 원본 범주형 직접 처리
- CV: 동일 입력과 모든 마스킹 변형을 묶는 `StratifiedGroupKFold(5)`
- 선택 기준: CV Brier Score 최소
- 보조 지표: Log Loss, AUC, Accuracy, Precision, Recall, F1
- 분류 임계값: 0.5

이 노트북의 마지막 셀은 선택된 파라미터와 데이터 서명을 로컬 아티팩트로 저장한다. 4단계는 이 파일을 읽어 동일한 모델로 앙상블을 구성한다.

## 0. 실행 환경

```bash
uv sync --project backend/notebooks --locked
uv run --project backend/notebooks --locked jupyter lab backend/notebooks/deal_model_phase3.ipynb
```

Random Search는 고정 시드를 사용한다. TabICL은 장치를 독점하도록 순차 실행한다.

In [1]:
import hashlib
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from catboost import CatBoostClassifier
from IPython import get_ipython
from IPython.display import display
from IPython.utils.capture import capture_output
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedGroupKFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from tabicl import TabICLClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 240)

current_dir = Path.cwd().resolve()
if current_dir.name == "notebooks":
    preprocessing_notebook = current_dir / "deal_data_preprocessing.ipynb"
elif current_dir.name == "backend" and (current_dir / "notebooks").is_dir():
    preprocessing_notebook = current_dir / "notebooks" / "deal_data_preprocessing.ipynb"
elif (current_dir / "backend" / "notebooks").is_dir():
    preprocessing_notebook = current_dir / "backend" / "notebooks" / "deal_data_preprocessing.ipynb"
else:
    raise RuntimeError("backend/notebooks 폴더를 찾을 수 없습니다.")

assert preprocessing_notebook.exists(), f"전처리 노트북이 없습니다: {preprocessing_notebook}"
ipython = get_ipython()
assert ipython is not None, "이 파일은 Jupyter에서 실행해야 합니다."
with capture_output():
    ipython.run_line_magic("run", str(preprocessing_notebook))

X_train_raw = globals()["X_train_raw"]
y_train = globals()["y_train"]
train_group_ids = globals()["train_group_ids"]
X_test_raw_sets = globals()["X_test_raw_sets"]
y_test = globals()["y_test"]
MODEL_FEATURE_NAMES = globals()["MODEL_FEATURE_NAMES"]
CATEGORY_VALUES = globals()["CATEGORY_VALUES"]

RANDOM_STATE = 1
CLASSIFICATION_THRESHOLD = 0.5
cv5 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_splits = list(cv5.split(X_train_raw, y_train, groups=train_group_ids))
repo_root = preprocessing_notebook.parents[2]
PHASE3_SELECTION_PATH = (
    repo_root / "backend" / "pipeline" / "artifacts" / "deal_phase3_selection.joblib"
)

assert X_train_raw.shape == (3130, 13)
assert len(np.unique(train_group_ids)) == 138
assert len(X_test_raw_sets) == 10
print(f"Train: {X_train_raw.shape}, 동일 입력 그룹: {len(np.unique(train_group_ids))}")
print(f"Test: {len(X_test_raw_sets)}세트 × {len(y_test)}행")

데이터 전처리 검증을 통과했습니다.
Train: (3130, 13), 동일 입력 그룹: 138
Test: 10세트 × 135행


### 해석

- 1단계의 원본 범주형 입력과 그룹 ID를 그대로 사용한다.
- 정밀 튜닝 중에도 같은 입력의 반복 행과 모든 마스킹 변형은 하나의 Fold에 유지된다.
- Test는 파라미터 탐색에 사용하지 않는다.

## 1. 공통 모델 생성·평가 과정

In [2]:
CV_SCORING = {
    "brier": "neg_brier_score",
    "logloss": "neg_log_loss",
    "auc": "roc_auc",
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
}
TEST_METRIC_COLUMNS = (
    "brier",
    "logloss",
    "auc",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "specificity",
    "fpr",
    "tn",
    "fp",
    "fn",
    "tp",
)


def make_one_hot_model(classifier):
    """13개 고정 범주를 원핫 인코딩한 뒤 전달하는 모델 파이프라인을 만든다."""
    encoder = OneHotEncoder(
        categories=[list(CATEGORY_VALUES[column]) for column in MODEL_FEATURE_NAMES],
        drop="first",
        handle_unknown="error",
        sparse_output=False,
        dtype=np.float32,
    )
    return Pipeline([("onehot", encoder), ("classifier", classifier)])


def positive_class_probability(estimator, X):
    won_index = list(estimator.classes_).index(1)
    return estimator.predict_proba(X)[:, won_index]


def evaluate_test_sets(estimator) -> pd.DataFrame:
    rows = []
    for set_name, X_test in X_test_raw_sets.items():
        probability = positive_class_probability(estimator, X_test)
        prediction = (probability >= CLASSIFICATION_THRESHOLD).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=[0, 1]).ravel()
        specificity = tn / (tn + fp)
        rows.append(
            {
                "test_set": set_name,
                "brier": brier_score_loss(y_test, probability),
                "logloss": log_loss(y_test, probability, labels=[0, 1]),
                "auc": roc_auc_score(y_test, probability),
                "accuracy": accuracy_score(y_test, prediction),
                "precision": precision_score(y_test, prediction, zero_division=0),
                "recall": recall_score(y_test, prediction, zero_division=0),
                "f1": f1_score(y_test, prediction, zero_division=0),
                "specificity": specificity,
                "fpr": 1 - specificity,
                "tn": int(tn),
                "fp": int(fp),
                "fn": int(fn),
                "tp": int(tp),
            }
        )
    return pd.DataFrame(rows)


def summarize_test_results(test_results: pd.DataFrame) -> pd.DataFrame:
    summary = test_results[list(TEST_METRIC_COLUMNS)].agg(["mean", "std", "min", "max"]).T
    summary.index.name = "metric"
    return summary


def search_result_table(search) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "params": search.cv_results_["params"],
            "cv_brier": -search.cv_results_["mean_test_brier"],
            "cv_brier_std": search.cv_results_["std_test_brier"],
            "cv_logloss": -search.cv_results_["mean_test_logloss"],
            "cv_auc": search.cv_results_["mean_test_auc"],
            "cv_accuracy": search.cv_results_["mean_test_accuracy"],
            "cv_precision": search.cv_results_["mean_test_precision"],
            "cv_recall": search.cv_results_["mean_test_recall"],
            "cv_f1": search.cv_results_["mean_test_f1"],
            "mean_fit_time": search.cv_results_["mean_fit_time"],
        }
    ).sort_values("cv_brier", ignore_index=True)


def fit_and_evaluate(search):
    search.fit(X_train_raw, y_train, groups=train_group_ids)
    test_results = evaluate_test_sets(search.best_estimator_)
    return search, test_results, summarize_test_results(test_results)


model_searches = {}
test_results_by_model = {}
test_summaries_by_model = {}
model_input_types = {}
print("공통 튜닝·평가 함수를 준비했습니다.")

공통 튜닝·평가 함수를 준비했습니다.


### 해석

최적 조합은 Brier로 선택하지만 같은 CV 후보에서 AUC·Accuracy·Precision·Recall·F1도 함께 기록한다.

## 2. LogisticRegression 정밀 튜닝

규제 강도와 클래스 가중치를 함께 비교한다.

In [3]:
model_logistic = make_one_hot_model(
    LogisticRegression(max_iter=3000, solver="lbfgs", random_state=RANDOM_STATE)
)
param_logistic = {
    "classifier__C": [0.01, 0.02, 0.03, 0.05, 0.08, 0.12, 0.2],
    "classifier__class_weight": [None, "balanced"],
}
search_logistic = GridSearchCV(
    model_logistic,
    param_logistic,
    scoring=CV_SCORING,
    refit="brier",
    cv=cv_splits,
    n_jobs=-1,
    error_score="raise",
)
search_logistic, test_results_logistic, test_summary_logistic = fit_and_evaluate(search_logistic)
model_searches["LogisticRegression"] = search_logistic
test_results_by_model["LogisticRegression"] = test_results_logistic
test_summaries_by_model["LogisticRegression"] = test_summary_logistic
model_input_types["LogisticRegression"] = "모델 내부 원핫 39개"
print(f"최적 파라미터: {search_logistic.best_params_}")
print(f"정밀 CV Brier: {-search_logistic.best_score_:.6f}")
display(search_result_table(search_logistic).round(6))
display(test_summary_logistic.round(6))

최적 파라미터: {'classifier__C': 0.03, 'classifier__class_weight': 'balanced'}
정밀 CV Brier: 0.197362


,params,cv_brier,cv_brier_std,cv_logloss,cv_auc,cv_accuracy,cv_precision,cv_recall,cv_f1,mean_fit_time
0,"{'classifier__C': 0.03, 'classifier__class_wei...",0.197362,0.018949,0.583127,0.768167,0.722158,0.714614,0.797697,0.743478,0.010456
1,"{'classifier__C': 0.03, 'classifier__class_wei...",0.197368,0.019469,0.583188,0.768105,0.722824,0.710734,0.807157,0.746305,0.010108
2,"{'classifier__C': 0.05, 'classifier__class_wei...",0.197564,0.020819,0.585213,0.765576,0.720190,0.712827,0.796307,0.741431,0.009554
3,"{'classifier__C': 0.05, 'classifier__class_wei...",0.197584,0.020234,0.585276,0.765604,0.718828,0.714912,0.787960,0.737680,0.010313
4,"{'classifier__C': 0.02, 'classifier__class_wei...",0.198107,0.018234,0.584220,0.769306,0.720630,0.706333,0.811653,0.746386,0.020360
5,"{'classifier__C': 0.02, 'classifier__class_wei...",0.198111,0.017779,0.584167,0.769321,0.719627,0.709169,0.802741,0.742947,0.010584
6,"{'classifier__C': 0.08, 'classifier__class_wei...",0.198476,0.021355,0.589511,0.763229,0.716850,0.714749,0.782799,0.734633,0.010318
7,"{'classifier__C': 0.08, 'classifier__class_wei...",0.198483,0.021932,0.589560,0.763165,0.716235,0.712537,0.786629,0.735355,0.009839
8,"{'classifier__C': 0.12, 'classifier__class_wei...",0.199579,0.022259,0.594436,0.760717,0.714944,0.712950,0.780316,0.732579,0.009328
9,"{'classifier__C': 0.12, 'classifier__class_wei...",0.199592,0.022885,0.594475,0.760799,0.714615,0.711236,0.783482,0.733367,0.010789


,mean,std,min,max
metric,,,,
brier,0.185205,0.006143,0.175825,0.195862
logloss,0.549889,0.014167,0.529830,0.571755
auc,0.806080,0.021546,0.768218,0.834284
accuracy,0.713333,0.025908,0.681481,0.748148
precision,0.737183,0.041733,0.696970,0.814815
recall,0.673529,0.022782,0.647059,0.720588
f1,0.703239,0.022079,0.671756,0.731343
specificity,0.753731,0.050369,0.701493,0.850746
fpr,0.246269,0.050369,0.149254,0.298507


### 해석 기준

강한 규제가 선택되면 현재 독립 그룹 수에서 복잡한 계수보다 단순한 확률 경계가 유리하다는 뜻이다.

## 3. MultinomialNB 정밀 튜닝

평활화가 다시 나빠지는 지점까지 `alpha` 범위를 넓힌다.

In [4]:
model_nb = make_one_hot_model(MultinomialNB())
param_nb = {
    "classifier__alpha": [10.0, 20.0, 40.0, 60.0, 75.0, 100.0, 150.0, 200.0],
    "classifier__fit_prior": [False, True],
}
search_nb = GridSearchCV(
    model_nb,
    param_nb,
    scoring=CV_SCORING,
    refit="brier",
    cv=cv_splits,
    n_jobs=-1,
    error_score="raise",
)
search_nb, test_results_nb, test_summary_nb = fit_and_evaluate(search_nb)
model_searches["MultinomialNB"] = search_nb
test_results_by_model["MultinomialNB"] = test_results_nb
test_summaries_by_model["MultinomialNB"] = test_summary_nb
model_input_types["MultinomialNB"] = "모델 내부 원핫 39개"
print(f"최적 파라미터: {search_nb.best_params_}")
print(f"정밀 CV Brier: {-search_nb.best_score_:.6f}")
display(search_result_table(search_nb).round(6))
display(test_summary_nb.round(6))

최적 파라미터: {'classifier__alpha': 75.0, 'classifier__fit_prior': False}


정밀 CV Brier: 0.198054


,params,cv_brier,cv_brier_std,cv_logloss,cv_auc,cv_accuracy,cv_precision,cv_recall,cv_f1,mean_fit_time
0,"{'classifier__alpha': 75.0, 'classifier__fit_p...",0.198054,0.026728,0.586041,0.764155,0.714208,0.688534,0.825282,0.746349,0.007682
1,"{'classifier__alpha': 60.0, 'classifier__fit_p...",0.198126,0.027610,0.587893,0.764290,0.715447,0.690614,0.823347,0.746752,0.007533
2,"{'classifier__alpha': 100.0, 'classifier__fit_...",0.198193,0.025403,0.584720,0.763743,0.711974,0.684257,0.829719,0.745959,0.007688
3,"{'classifier__alpha': 75.0, 'classifier__fit_p...",0.198346,0.027382,0.586823,0.764155,0.712004,0.683683,0.831009,0.746268,0.007431
4,"{'classifier__alpha': 60.0, 'classifier__fit_p...",0.198393,0.028285,0.588622,0.764290,0.712948,0.686449,0.827179,0.746058,0.006958
5,"{'classifier__alpha': 40.0, 'classifier__fit_p...",0.198502,0.028976,0.592592,0.764450,0.717681,0.694208,0.820786,0.747682,0.007444
6,"{'classifier__alpha': 100.0, 'classifier__fit_...",0.198518,0.026024,0.585568,0.763743,0.711360,0.682457,0.834196,0.746755,0.007730
7,"{'classifier__alpha': 40.0, 'classifier__fit_p...",0.198724,0.029678,0.593223,0.764450,0.715151,0.690003,0.825243,0.747071,0.007742
8,"{'classifier__alpha': 150.0, 'classifier__fit_...",0.199079,0.023104,0.585498,0.763121,0.708420,0.678925,0.833511,0.744522,0.008933
9,"{'classifier__alpha': 150.0, 'classifier__fit_...",0.199449,0.023666,0.586423,0.763121,0.708092,0.676894,0.840508,0.746151,0.008253


,mean,std,min,max
metric,,,,
brier,0.181453,0.007900,0.169123,0.193061
logloss,0.537153,0.019336,0.510048,0.573601
auc,0.800373,0.020257,0.772169,0.829675
accuracy,0.721481,0.022684,0.688889,0.755556
precision,0.720401,0.026165,0.680000,0.769231
recall,0.732353,0.037203,0.661765,0.779412
f1,0.725737,0.023885,0.681818,0.762590
specificity,0.710448,0.039300,0.641791,0.776119
fpr,0.289552,0.039300,0.223881,0.358209


### 해석 기준

최적 `alpha`가 범위 중앙에 있으면 평활화 최적점이 탐색 범위 안에 들어온 것으로 본다.

## 4. ExtraTrees 정밀 튜닝

깊이, 잎 크기, 입력 비율, 트리 수를 함께 세분화한다.

In [5]:
model_extratrees = make_one_hot_model(ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=1))
param_extratrees = {
    "classifier__n_estimators": [300, 600],
    "classifier__max_depth": [4, 5, 6, 7, 8],
    "classifier__min_samples_leaf": [3, 4, 5, 6],
    "classifier__max_features": ["sqrt", 0.25, 0.4],
}
search_extratrees = GridSearchCV(
    model_extratrees,
    param_extratrees,
    scoring=CV_SCORING,
    refit="brier",
    cv=cv_splits,
    n_jobs=-1,
    error_score="raise",
)
search_extratrees, test_results_extratrees, test_summary_extratrees = fit_and_evaluate(
    search_extratrees
)
model_searches["ExtraTrees"] = search_extratrees
test_results_by_model["ExtraTrees"] = test_results_extratrees
test_summaries_by_model["ExtraTrees"] = test_summary_extratrees
model_input_types["ExtraTrees"] = "모델 내부 원핫 39개"
print(f"최적 파라미터: {search_extratrees.best_params_}")
print(f"정밀 CV Brier: {-search_extratrees.best_score_:.6f}")
display(search_result_table(search_extratrees).head(30).round(6))
display(test_summary_extratrees.round(6))

최적 파라미터: {'classifier__max_depth': 8, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 6, 'classifier__n_estimators': 300}
정밀 CV Brier: 0.198232


,params,cv_brier,cv_brier_std,cv_logloss,cv_auc,cv_accuracy,cv_precision,cv_recall,cv_f1,mean_fit_time
0,"{'classifier__max_depth': 8, 'classifier__max_...",0.198232,0.021283,0.586317,0.766946,0.721902,0.701862,0.813086,0.749056,0.271113
1,"{'classifier__max_depth': 8, 'classifier__max_...",0.198300,0.021043,0.587207,0.767073,0.722875,0.702845,0.814356,0.750035,0.534302
2,"{'classifier__max_depth': 7, 'classifier__max_...",0.198401,0.020466,0.586169,0.767574,0.725795,0.702899,0.822078,0.754142,0.246638
3,"{'classifier__max_depth': 7, 'classifier__max_...",0.198562,0.020203,0.586910,0.767676,0.726451,0.704424,0.820808,0.754302,0.486412
4,"{'classifier__max_depth': 7, 'classifier__max_...",0.198741,0.020424,0.586798,0.767048,0.723900,0.701443,0.818933,0.752079,0.247901
5,"{'classifier__max_depth': 7, 'classifier__max_...",0.198834,0.020261,0.587031,0.766128,0.726154,0.702734,0.823368,0.754705,0.246295
6,"{'classifier__max_depth': 7, 'classifier__max_...",0.198835,0.020117,0.587411,0.766493,0.725816,0.703466,0.820827,0.753795,0.485502
7,"{'classifier__max_depth': 8, 'classifier__max_...",0.198929,0.021329,0.588763,0.764927,0.723183,0.703402,0.813731,0.750127,0.544694
8,"{'classifier__max_depth': 6, 'classifier__max_...",0.198981,0.019237,0.586854,0.768791,0.726523,0.702776,0.824718,0.755293,0.224277
9,"{'classifier__max_depth': 8, 'classifier__max_...",0.198992,0.021435,0.588283,0.764637,0.721246,0.700799,0.813711,0.748741,0.285426


,mean,std,min,max
metric,,,,
brier,0.181914,0.005569,0.173460,0.189823
logloss,0.543751,0.013454,0.525731,0.567743
auc,0.812028,0.020561,0.773266,0.836479
accuracy,0.742963,0.020376,0.711111,0.777778
precision,0.705081,0.019280,0.670588,0.734177
recall,0.842647,0.021977,0.823529,0.897059
f1,0.767612,0.017448,0.745098,0.802632
specificity,0.641791,0.030669,0.582090,0.686567
fpr,0.358209,0.030669,0.313433,0.417910


### 해석 기준

트리 수만 늘려도 Brier가 거의 같으면 더 적은 트리 수가 운영상 충분하다.

## 5. CatBoost 정밀 튜닝

원본 범주형 입력에서 학습 횟수, 깊이, 학습률, L2 규제, 무작위 강도를 탐색한다.

In [6]:
model_catboost = CatBoostClassifier(
    cat_features=tuple(MODEL_FEATURE_NAMES),
    loss_function="Logloss",
    verbose=False,
    allow_writing_files=False,
    random_seed=RANDOM_STATE,
    thread_count=1,
)
param_catboost = {
    "iterations": [200, 350, 500, 700, 900],
    "depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.02, 0.03, 0.05],
    "l2_leaf_reg": [3.0, 5.0, 7.0, 10.0, 15.0, 20.0],
    "random_strength": [0.5, 1.0, 1.5, 2.0],
}
search_catboost = RandomizedSearchCV(
    model_catboost,
    param_catboost,
    n_iter=80,
    scoring=CV_SCORING,
    refit="brier",
    cv=cv_splits,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    error_score="raise",
)
search_catboost, test_results_catboost, test_summary_catboost = fit_and_evaluate(search_catboost)
model_searches["CatBoost"] = search_catboost
test_results_by_model["CatBoost"] = test_results_catboost
test_summaries_by_model["CatBoost"] = test_summary_catboost
model_input_types["CatBoost"] = "원본 범주형 13개"
print(f"최적 파라미터: {search_catboost.best_params_}")
print(f"정밀 CV Brier: {-search_catboost.best_score_:.6f}")
display(search_result_table(search_catboost).head(30).round(6))
display(test_summary_catboost.round(6))

최적 파라미터: {'random_strength': 0.5, 'learning_rate': 0.01, 'l2_leaf_reg': 7.0, 'iterations': 200, 'depth': 4}
정밀 CV Brier: 0.194832


,params,cv_brier,cv_brier_std,cv_logloss,cv_auc,cv_accuracy,cv_precision,cv_recall,cv_f1,mean_fit_time
0,"{'random_strength': 0.5, 'learning_rate': 0.01...",0.194832,0.022914,0.581544,0.767269,0.715609,0.705656,0.790542,0.737052,0.705390
1,"{'random_strength': 0.5, 'learning_rate': 0.01...",0.194952,0.024752,0.585844,0.766718,0.714606,0.707233,0.785382,0.734720,1.292588
2,"{'random_strength': 1.0, 'learning_rate': 0.01...",0.195000,0.024562,0.583048,0.770940,0.717545,0.699056,0.803968,0.742331,0.940535
3,"{'random_strength': 2.0, 'learning_rate': 0.03...",0.195087,0.026523,0.588676,0.771789,0.716203,0.706666,0.787335,0.736193,0.708689
4,"{'random_strength': 0.5, 'learning_rate': 0.01...",0.195134,0.024244,0.588643,0.768532,0.715855,0.706505,0.787901,0.736009,1.402763
5,"{'random_strength': 1.5, 'learning_rate': 0.01...",0.195524,0.025171,0.587832,0.768273,0.714964,0.706100,0.787317,0.735649,2.167819
6,"{'random_strength': 0.5, 'learning_rate': 0.01...",0.195768,0.025995,0.589727,0.764694,0.716195,0.709989,0.784170,0.735273,1.661416
7,"{'random_strength': 1.5, 'learning_rate': 0.01...",0.195883,0.025503,0.587225,0.768420,0.719113,0.708947,0.792377,0.739638,1.380905
8,"{'random_strength': 1.0, 'learning_rate': 0.02...",0.195926,0.026585,0.593058,0.768960,0.713653,0.706767,0.782879,0.733563,1.398256
9,"{'random_strength': 0.5, 'learning_rate': 0.01...",0.195933,0.021897,0.582915,0.768710,0.718671,0.694258,0.813847,0.746730,0.559062


,mean,std,min,max
metric,,,,
brier,0.186809,0.006452,0.174944,0.194484
logloss,0.555022,0.014827,0.529088,0.574133
auc,0.790222,0.023035,0.757902,0.816396
accuracy,0.740741,0.024937,0.703704,0.792593
precision,0.702151,0.023516,0.662791,0.743902
recall,0.844118,0.028751,0.794118,0.897059
f1,0.766377,0.021672,0.739726,0.813333
specificity,0.635821,0.038665,0.567164,0.686567
fpr,0.364179,0.038665,0.313433,0.432836


### 해석 기준

원핫 입력에서 얻은 이전 파라미터를 재사용하지 않는다. 범주형 직접 처리 조건에서 새로 선택된 값만 다음 단계에 전달한다.

## 6. TabICL 정밀 튜닝

원본 범주형 입력에서 내부 앙상블 수, 정규화, 확률 온도를 비교한다.

In [7]:
if torch.backends.mps.is_available():
    tabicl_device = "mps"
elif torch.cuda.is_available():
    tabicl_device = "cuda"
else:
    tabicl_device = "cpu"

model_tabicl = TabICLClassifier(
    batch_size=8,
    kv_cache=False,
    allow_auto_download=True,
    device=tabicl_device,
    use_fa3="auto",
    offload_mode="auto",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=False,
)
param_tabicl = {
    "n_estimators": [8, 16],
    "norm_methods": [None, "quantile"],
    "feat_shuffle_method": ["latin"],
    "average_logits": [True],
    "softmax_temperature": [0.9, 1.0, 1.1],
}
search_tabicl = GridSearchCV(
    model_tabicl,
    param_tabicl,
    scoring=CV_SCORING,
    refit="brier",
    cv=cv_splits,
    n_jobs=1,
    error_score="raise",
)
search_tabicl, test_results_tabicl, test_summary_tabicl = fit_and_evaluate(search_tabicl)
model_searches["TabICL"] = search_tabicl
test_results_by_model["TabICL"] = test_results_tabicl
test_summaries_by_model["TabICL"] = test_summary_tabicl
model_input_types["TabICL"] = "원본 범주형 13개"
print(f"장치: {tabicl_device}")
print(f"최적 파라미터: {search_tabicl.best_params_}")
print(f"정밀 CV Brier: {-search_tabicl.best_score_:.6f}")
display(search_result_table(search_tabicl).round(6))
display(test_summary_tabicl.round(6))

장치: mps
최적 파라미터: {'average_logits': True, 'feat_shuffle_method': 'latin', 'n_estimators': 8, 'norm_methods': 'quantile', 'softmax_temperature': 1.1}
정밀 CV Brier: 0.213916


,params,cv_brier,cv_brier_std,cv_logloss,cv_auc,cv_accuracy,cv_precision,cv_recall,cv_f1,mean_fit_time
0,"{'average_logits': True, 'feat_shuffle_method'...",0.213916,0.030760,0.689077,0.737885,0.694599,0.695493,0.763160,0.715965,0.181214
1,"{'average_logits': True, 'feat_shuffle_method'...",0.214179,0.031183,0.690775,0.737972,0.696168,0.696362,0.763785,0.717136,0.188812
2,"{'average_logits': True, 'feat_shuffle_method'...",0.215644,0.032103,0.709453,0.737885,0.694599,0.695493,0.763160,0.715965,0.182325
3,"{'average_logits': True, 'feat_shuffle_method'...",0.215938,0.032528,0.711427,0.737972,0.696168,0.696362,0.763785,0.717136,0.181465
4,"{'average_logits': True, 'feat_shuffle_method'...",0.218041,0.033636,0.736716,0.737885,0.694599,0.695493,0.763160,0.715965,0.183255
5,"{'average_logits': True, 'feat_shuffle_method'...",0.218367,0.034066,0.739039,0.737972,0.696168,0.696362,0.763785,0.717136,0.181396
6,"{'average_logits': True, 'feat_shuffle_method'...",0.223154,0.025517,0.742291,0.724965,0.690928,0.682419,0.760370,0.713005,0.307310
7,"{'average_logits': True, 'feat_shuffle_method'...",0.223802,0.025446,0.743732,0.723348,0.689985,0.681386,0.761601,0.712742,0.187558
8,"{'average_logits': True, 'feat_shuffle_method'...",0.225753,0.026475,0.773395,0.724965,0.690928,0.682419,0.760370,0.713005,0.214841
9,"{'average_logits': True, 'feat_shuffle_method'...",0.226417,0.026401,0.774910,0.723348,0.689985,0.681386,0.761601,0.712742,0.181931


,mean,std,min,max
metric,,,,
brier,0.177992,0.005323,0.171616,0.186647
logloss,0.536877,0.014871,0.514283,0.562750
auc,0.806629,0.015521,0.784021,0.828797
accuracy,0.730370,0.029876,0.688889,0.770370
precision,0.714184,0.025152,0.666667,0.741379
recall,0.776471,0.068206,0.632353,0.852941
f1,0.742526,0.036429,0.682540,0.786207
specificity,0.683582,0.044942,0.611940,0.776119
fpr,0.316418,0.044942,0.223881,0.388060


### 해석 기준

16개 내부 추정기가 8개보다 의미 있게 좋아야 추가 추론 비용을 감수할 근거가 생긴다.

## 7. 정밀 튜닝 결과 비교

In [8]:
comparison_rows = []
for model_name, search in model_searches.items():
    summary = test_summaries_by_model[model_name]
    comparison_rows.append(
        {
            "model": model_name,
            "input": model_input_types[model_name],
            "cv_brier": -search.best_score_,
            "cv_brier_std": search.cv_results_["std_test_brier"][search.best_index_],
            "cv_logloss": -search.cv_results_["mean_test_logloss"][search.best_index_],
            "cv_auc": search.cv_results_["mean_test_auc"][search.best_index_],
            "cv_accuracy": search.cv_results_["mean_test_accuracy"][search.best_index_],
            "cv_precision": search.cv_results_["mean_test_precision"][search.best_index_],
            "cv_recall": search.cv_results_["mean_test_recall"][search.best_index_],
            "cv_f1": search.cv_results_["mean_test_f1"][search.best_index_],
            "test_brier_mean": summary.loc["brier", "mean"],
            "test_auc_mean": summary.loc["auc", "mean"],
            "test_accuracy_mean": summary.loc["accuracy", "mean"],
            "test_precision_mean": summary.loc["precision", "mean"],
            "test_recall_mean": summary.loc["recall", "mean"],
            "test_f1_mean": summary.loc["f1", "mean"],
            "test_fp_mean": summary.loc["fp", "mean"],
            "test_fn_mean": summary.loc["fn", "mean"],
            "best_params": search.best_params_,
        }
    )

comparison = pd.DataFrame(comparison_rows).sort_values("cv_brier", ignore_index=True)
display(comparison.round(6))
print(f"정밀 CV Brier 기준 1순위: {comparison.iloc[0]['model']}")

,model,input,cv_brier,cv_brier_std,cv_logloss,cv_auc,cv_accuracy,cv_precision,cv_recall,cv_f1,test_brier_mean,test_auc_mean,test_accuracy_mean,test_precision_mean,test_recall_mean,test_f1_mean,test_fp_mean,test_fn_mean,best_params
0,CatBoost,원본 범주형 13개,0.194832,0.022914,0.581544,0.767269,0.715609,0.705656,0.790542,0.737052,0.186809,0.790222,0.740741,0.702151,0.844118,0.766377,24.4,10.6,"{'random_strength': 0.5, 'learning_rate': 0.01..."
1,LogisticRegression,모델 내부 원핫 39개,0.197362,0.018949,0.583127,0.768167,0.722158,0.714614,0.797697,0.743478,0.185205,0.806080,0.713333,0.737183,0.673529,0.703239,16.5,22.2,"{'classifier__C': 0.03, 'classifier__class_wei..."
2,MultinomialNB,모델 내부 원핫 39개,0.198054,0.026728,0.586041,0.764155,0.714208,0.688534,0.825282,0.746349,0.181453,0.800373,0.721481,0.720401,0.732353,0.725737,19.4,18.2,"{'classifier__alpha': 75.0, 'classifier__fit_p..."
3,ExtraTrees,모델 내부 원핫 39개,0.198232,0.021283,0.586317,0.766946,0.721902,0.701862,0.813086,0.749056,0.181914,0.812028,0.742963,0.705081,0.842647,0.767612,24.0,10.7,"{'classifier__max_depth': 8, 'classifier__max_..."
4,TabICL,원본 범주형 13개,0.213916,0.030760,0.689077,0.737885,0.694599,0.695493,0.763160,0.715965,0.177992,0.806629,0.730370,0.714184,0.776471,0.742526,21.2,15.2,"{'average_logits': True, 'feat_shuffle_method'..."


정밀 CV Brier 기준 1순위: CatBoost


### 해석 기준

- Brier 한 값만 보지 않고 AUC, Accuracy, Precision, Recall, F1, FP·FN을 함께 확인한다.
- CV 차이가 표준편차보다 작으면 소수점 순위보다 모델 단순성과 앙상블 기여도를 다음 단계에서 확인한다.
- Test 결과에 맞춰 파라미터를 다시 바꾸지 않는다.

In [9]:
def training_data_signature() -> str:
    """4단계가 정확히 같은 데이터와 행 순서를 사용하도록 서명을 만든다."""
    parts = (
        pd.util.hash_pandas_object(X_train_raw.astype("string"), index=True)
        .to_numpy(dtype=np.uint64)
        .tobytes(),
        np.asarray(y_train, dtype=np.int8).tobytes(),
        np.asarray(train_group_ids, dtype=np.uint64).tobytes(),
    )
    return hashlib.sha256(b"".join(parts)).hexdigest()


expected_models = {
    "LogisticRegression",
    "MultinomialNB",
    "ExtraTrees",
    "CatBoost",
    "TabICL",
}
assert set(model_searches) == expected_models
assert comparison["model"].nunique() == len(expected_models)
assert comparison.drop(columns=["model", "input", "best_params"]).notna().all().all()
assert all(len(results) == 10 for results in test_results_by_model.values())

selection_artifact = {
    "schema_version": 2,
    "data_signature": training_data_signature(),
    "model_feature_names": list(MODEL_FEATURE_NAMES),
    "best_params": {
        model_name: dict(search.best_params_) for model_name, search in model_searches.items()
    },
    "candidate_params": {
        model_name: [dict(params) for params in search.cv_results_["params"]]
        for model_name, search in model_searches.items()
    },
    "cv_brier": {
        model_name: float(-search.best_score_) for model_name, search in model_searches.items()
    },
}
PHASE3_SELECTION_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(selection_artifact, PHASE3_SELECTION_PATH)
reloaded_artifact = joblib.load(PHASE3_SELECTION_PATH)
assert reloaded_artifact["data_signature"] == selection_artifact["data_signature"]
assert set(reloaded_artifact["best_params"]) == expected_models
print("3단계 정밀 튜닝 검증을 통과했습니다.")
print(f"4단계 입력 아티팩트: {PHASE3_SELECTION_PATH}")

3단계 정밀 튜닝 검증을 통과했습니다.
4단계 입력 아티팩트: backend/pipeline/artifacts/deal_phase3_selection.joblib


### 해석

- 선택된 다섯 모델의 최적 파라미터, 탐색한 전체 후보 목록, 학습 데이터 서명을 로컬 아티팩트에 저장했다.
- 모델 자체나 원본 데이터는 저장하지 않는다. 4단계는 아티팩트의 파라미터로 모델을 다시 만들고 같은 데이터인지 서명으로 확인한다.
- 아티팩트 경로는 저장소의 Git 제외 대상이므로 실행 결과가 코드 변경으로 섞이지 않는다.